[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/01_%E3%83%87%E3%83%BC%E3%82%BF%E3%81%AE%E7%A8%AE%E9%A1%9E%E3%81%A8%E5%BA%A6%E6%95%B0%E5%88%86%E5%B8%83.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計4級-01. データの種類と度数分布表

**できるようになること**
- データを質的/量的・4つの尺度（名義・順序・間隔・比例）に見分けられる
- 度数分布表（度数・相対度数・累積度数）を作れる
- ヒストグラムで分布の形を読める

**前提**：`01_Python基礎`（pandas入門まで）　/　**所要**：約25分　/　**レベル**：統計検定4級相当

## 1. データの種類

| 大分類 | 小分類 | 説明 | 例 |
|---|---|---|---|
| **質的データ** | 名義尺度 | 区別だけ。順番に意味なし | 血液型・クラス |
| | 順序尺度 | 順番に意味あり | 順位・満足度(5段階) |
| **量的データ** | 間隔尺度 | 差に意味。0は便宜的 | 気温(℃)・西暦 |
| | 比例尺度 | 差にも比にも意味。0は無 | 身長・点数・人数 |

> クイズ：「テストの点数」「好きな教科」「気温」はそれぞれどの尺度？

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画

# 日本語が文字化けするときは次の1行を有効にしてください
# plt.rcParams['font.family'] = 'Hiragino Sans'  # Macの場合
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
df = pd.read_csv('../data/students_scores.csv')   # 生徒データを読み込む
df.head()                        # 先頭5行を確認

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

各列がどんなデータか確認しましょう。`クラス` は文字列（質的）、`数学` は数値（量的）です。

In [ ]:
df.dtypes   # 各列のデータ型（数値か文字列か）を確認

## 2. 度数分布表をつくる

数学の点数を「階級（10点ごとの区間）」に分けて、各区間に何人いるか数えます。
これを **度数** といいます。

In [ ]:
bins = [0, 20, 40, 60, 80, 100]                        # 階級の境目
labels = ['0-19', '20-39', '40-59', '60-79', '80-100']  # 各階級の名前
df['階級'] = pd.cut(df['数学'], bins=bins, right=False, labels=labels)  # 数学の点を階級に振り分け
freq = df['階級'].value_counts().sort_index()          # 階級ごとの人数（度数）
table = pd.DataFrame({'度数': freq})                   # 度数を表にする
table['相対度数'] = (table['度数'] / table['度数'].sum()).round(3)  # 全体に対する割合
table['累積度数'] = table['度数'].cumsum()             # 下から積み上げた合計
table

**読み取りのポイント**
- 度数：その階級の人数
- 相対度数：全体に対する割合（合計1）
- 累積度数：その階級までの人数の合計

## 3. ヒストグラムで見える化

In [ ]:
plt.figure(figsize=(7, 4))                  # グラフの大きさを指定
plt.hist(df['数学'], bins=bins, edgecolor='white')  # ヒストグラム（階級ごとの人数）
plt.xlabel('数学の点数')                    # x軸ラベル
plt.ylabel('人数')                          # y軸ラベル
plt.title('数学の点数の分布')               # タイトル
plt.show()                                  # 表示

> ヒストグラムは「棒グラフ」と違い、棒どうしがくっつきます（連続した量を表すため）。

## 4. 累積相対度数で「上位◯%」を読む

相対度数を上から積み上げると「下から数えて何%にあたるか」がわかります。順位や分位（上位◯%）の感覚はここから生まれます。

In [ ]:
table['累積相対度数'] = table['相対度数'].cumsum().round(3)  # 相対度数を上から積み上げる
table  # 度数・相対度数・累積度数・累積相対度数を一覧で確認

> 累積相対度数が初めて 0.5 を超える階級が、おおまかな**中央値の階級**です。（このあと 3級で「中央値を含む階級」として再登場します）

> **よくある間違い**：気温(℃)の「0」は“無い”を意味しない（**間隔尺度**）。一方で点数・人数の「0」は“無い”を表す（**比例尺度**）。0の意味で尺度が変わります。

> **よくある間違い**：質的データそのものに**平均は使えない**。ただし 0/1 に数値化すれば**平均＝割合**になる（例：合格=1で合格率）。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 数学が80点以上の人数を ans に入れよう
ans = None   # 例: (df['数学'] >= 80).sum()
_check('Q1 80点以上の人数', ans, int((df['数学'] >= 80).sum()))

In [ ]:
# Q2: 80点以上の人の割合（相対度数, 0〜1）を ans に
ans = None   # 例: (df['数学'] >= 80).mean()
_check('Q2 相対度数', ans, float((df['数学'] >= 80).mean()))

In [ ]:
# Q3: 数学が60点未満の人の割合（累積相対度数, 0〜1）を ans に
ans = None   # 例: (df['数学'] < 60).mean()
_check('Q3 60点未満の割合', ans, float((df['数学'] < 60).mean()))

---
## 練習問題（4級スタイル）

**問1.** 次のデータの尺度を答えよ：(a) 都道府県名 (b) マラソンの順位 (c) 体重 (d) 摂氏温度

**問2.** `英語` の点数で、階級 `[0,20,40,60,80,100]` の度数分布表を作ろう。

**問3.** `英語` のヒストグラムを描き、いちばん人数が多い階級を答えよう。

**問4.** `勉強時間` を階級 `[0,1,2,3,4,5]` で度数分布表にし、**（階級値×度数）の合計 ÷ 人数** で平均を概算しよう。実際の `df['勉強時間'].mean()` とどれくらい近い？（＝度数分布表だけが手元にあるときの平均の求め方）

In [ ]:
# 問2・問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[01_データの種類と度数分布 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/02_%E7%B5%B1%E8%A8%88_4%E7%B4%9A/01_%E3%83%87%E3%83%BC%E3%82%BF%E3%81%AE%E7%A8%AE%E9%A1%9E%E3%81%A8%E5%BA%A6%E6%95%B0%E5%88%86%E5%B8%83.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 質的/量的 | 種類を表す/数量を表す |
| 名義・順序 | 区別だけ/順番に意味 |
| 間隔・比例 | 0が便宜的/0が“無い” |
| 度数・相対度数 | 各階級の個数/全体に対する割合 |
| 累積度数 | その階級までの合計 |
| 階級・階級値 | 区間/区間の代表値 |
| ヒストグラム | 量の分布を棒で表す図（棒がくっつく） |

In [ ]:
# チートシート（度数分布）
pd.cut(df['数学'], bins=[0,20,40,60,80,100])   # 階級に分ける
df['数学'].value_counts()                       # 値ごとの個数
plt.hist(df['数学'], bins=10)                   # ヒストグラム

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""       # ← あなたの名前（例: 山田）
LOG_URL = ""    # ← 記録用URL（配布時に設定）
LOG_TOKEN = ""  # ← 記録用トークン（配布時に設定）
NOTEBOOK = "02_統計_4級/01_データの種類と度数分布"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")